# nb_lib_logging
Logging for every job. Requires `%run nb_config` first.

Two kinds of logging:
1. **`get_logger`**: Python `logging` lines in notebook / pipeline output (for humans, while it runs)
2. **`RunLogger`**: one row per job run in `ops.run_log` (for querying history afterwards)

```python
with RunLogger("nb_03_silver", run_date, parent_run_id) as run:
    run.log.info("processing orders")
    run.metric("orders_merged", 1532)
```

In [ ]:
import json
import logging
import sys
import time
import traceback
import uuid

_LOG_FORMAT = "%(asctime)s | %(levelname)-7s | %(name)s | run=%(run_id)s | %(message)s"


class _RunIdFilter(logging.Filter):
    """Adds run_id to every record so it appears in each log line."""

    def __init__(self, run_id: str):
        super().__init__()
        self.run_id = run_id

    def filter(self, record):
        if not hasattr(record, "run_id"):
            record.run_id = self.run_id
        return True


def get_logger(name: str, run_id: str = "-", level: str = None, stream=None) -> logging.Logger:
    """Logger 'retail.<name>' writing to stdout in a consistent, greppable format.

    Handlers are reset on every call, so re-running %run in the same session never
    produces duplicate lines.
    """
    logger = logging.getLogger(f"{PROJECT}.{name}")
    logger.setLevel(level or LOG_LEVEL)
    logger.propagate = False
    for h in list(logger.handlers):
        logger.removeHandler(h)

    handler = logging.StreamHandler(stream or sys.stdout)
    formatter = logging.Formatter(_LOG_FORMAT, datefmt="%Y-%m-%d %H:%M:%S")
    formatter.converter = time.gmtime  # log timestamps in UTC
    handler.setFormatter(formatter)
    handler.addFilter(_RunIdFilter(run_id))
    logger.addHandler(handler)
    return logger


# Quieten noisy libraries in notebook output
for _noisy in ("py4j", "azure", "urllib3"):
    logging.getLogger(_noisy).setLevel(logging.WARNING)

In [ ]:
class RunLogger:
    """Context manager that logs START/END and writes one row to ops.run_log.

    * status = SUCCEEDED, or FAILED if an exception escapes the `with` block
    * exceptions are re-raised, so the notebook / pipeline still fails
    * writing the log row never hides the original error
    """

    def __init__(self, job_name: str, run_date=None, parent_run_id: str = None,
                 log_table: str = None):
        self.run_id = str(uuid.uuid4())
        self.job_name = job_name
        self.run_date = resolve_run_date(run_date)
        self.parent_run_id = parent_run_id or None
        self.log_table = log_table or tbl("ops", "run_log")
        self.metrics = {}
        self.status = "RUNNING"
        self.log = get_logger(job_name, run_id=self.run_id[:8])

    def metric(self, key: str, value):
        """Record a metric (stored as JSON in run_log.metrics_json)."""
        self.metrics[key] = value
        self.log.info("metric %s=%s", key, value)

    def __enter__(self):
        self.start_ts = utc_now()
        self.log.info("START job=%s run_date=%s env=%s parent_run_id=%s",
                      self.job_name, self.run_date, ENV, self.parent_run_id)
        return self

    def __exit__(self, exc_type, exc, tb):
        end_ts = utc_now()
        duration = (end_ts - self.start_ts).total_seconds()
        self.status = "SUCCEEDED" if exc_type is None else "FAILED"
        error = None
        if exc_type is not None:
            error = "".join(traceback.format_exception(exc_type, exc, tb))[-4000:]
            self.log.error("FAILED: %s: %s", exc_type.__name__, exc)

        row = [(self.run_id, self.parent_run_id, self.job_name, ENV, self.run_date,
                self.status, self.start_ts, end_ts, float(duration),
                json.dumps(self.metrics, default=str), error)]
        try:
            (spark.createDataFrame(row, OPS_TABLES["run_log"])
                .write.format("delta").mode("append").saveAsTable(self.log_table))
        except Exception as log_exc:
            self.log.warning("could not write run log to %s: %s", self.log_table, log_exc)

        self.log.info("END status=%s duration=%.1fs", self.status, duration)
        return False  # never swallow the exception

    def exit_value(self) -> str:
        """JSON summary for notebookutils.notebook.exit(): visible in pipeline output."""
        return json.dumps({"run_id": self.run_id, "job": self.job_name,
                           "status": self.status, "metrics": self.metrics}, default=str)


print("[nb_lib_logging] loaded")